# AdventureWorks — Exercise 2: Data Architecture & Schema Design

**Course:** Data Engineering  
**Notebook:** 02 — Schema Design, OLTP, OLAP, and Star Schema  
**Assistant:** Antigravity  

---

## What This Notebook Covers

| # | Section | Purpose |
|---|---------|----------|
| 1 | Setup & Load Raw Staging | Import modules, read Parquet from staging/raw/ |
| 2 | Data Validation | Schema, NULL, duplicate, and outlier checks |
| 3 | Data Transformation | Clean Product, Customer, and Sales tables |
| 4 | Aggregations | Sales by product, customer, date |
| 5 | OLTP Schema Design | Normalized operational model |
| 6 | Star Schema (OLAP) | DimDate, DimProduct, DimCustomer, FactSales |
| 7 | OLAP Operations | Roll-up, Drill-down, Slice, Dice |
| 8 | Summary | What was accomplished |

> **Full Refresh Architecture:** All data is rebuilt from source each run.


---
## Section 1 — Setup & Load from Raw Staging

**Objective:**  
Import all project modules and load the raw Parquet files written by Step 1.

> We read from `staging/raw/` rather than re-reading CSVs — the raw layer is the contract  
> between extraction and transformation. This is faster and preserves exact source state.


In [ ]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Add src/ to path
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from config import (
    STAGING_RAW, STAGING_VALID, STAGING_INVALID,
    STAGING_DUPLICATES, STAGING_OUTLIERS, ensure_directories,
)
from ingestion    import extract_all
from validation   import validate_all, print_validation_report
from transformation import transform_all
from olap import (
    build_dim_date, build_dim_product, build_dim_customer,
    build_fact_sales, build_olap,
    olap_rollup, olap_drilldown, olap_slice, olap_dice,
)

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.2f}'.format)
ensure_directories()

print('Modules loaded.')
print(f'Project Root: {PROJECT_ROOT}')

In [ ]:
# ── Extract all four source tables ────────────────────────────
# Full Refresh: always read from the source CSV files
print('Extracting source data (Full Refresh) ...')
raw = extract_all()

print(f'\n  Product          : {len(raw["product"]):>7,} rows')
print(f'  Customer         : {len(raw["customer"]):>7,} rows')
print(f'  SalesOrderHeader : {len(raw["salesorderheader"]):>7,} rows')
print(f'  SalesOrderDetail : {len(raw["salesorderdetail"]):>7,} rows')
print('\nExtraction complete.')

**Explanation:**  
`extract_all()` reads all 4 source CSVs using the settings in `config.py`.  
This is Step 1 of our pipeline — every run starts fresh from the source files.


---
## Section 2 — Data Validation

**Objective:**  
Run four validation checks on all raw tables:
1. **Schema Validation** — are all expected columns present?
2. **Null Validation** — are key columns (ProductID, CustomerID, etc.) populated?
3. **Duplicate Detection** — are primary keys unique?
4. **Outlier Detection** — are numeric values within expected ranges (IQR method)?

**What is IQR?**  
The Inter-Quartile Range (IQR = Q3 − Q1) defines the "middle 50%" of data.  
Values below `Q1 − 1.5 × IQR` or above `Q3 + 1.5 × IQR` are flagged as outliers.


In [ ]:
print('Running validation pipeline ...\n')
val_result = validate_all(raw)
print_validation_report(val_result)

In [ ]:
# Inspect product outliers in detail
print('\n── Product Outlier Detail ──')
print('(Records where StandardCost, ListPrice, or Weight is outside IQR bounds)\n')

prod_valid = val_result['valid']['product']
if '_is_outlier' in prod_valid.columns:
    outliers = prod_valid[prod_valid['_is_outlier'] == True]
    show_cols = ['ProductID', 'Name', 'Color', 'StandardCost', 'ListPrice', 'Weight']
    available = [c for c in show_cols if c in outliers.columns]
    display(outliers[available].head(10))
    print(f'\nTotal outlier records: {len(outliers)}')
    print('Note: Outliers are flagged but NOT removed from the valid dataset.')
    print('They are saved separately in staging/outliers/')

In [ ]:
# Show what files were written to staging folders
print('\n── Staging Files Written by Validation ──\n')
for folder_name, folder_path in [
    ('valid/',      STAGING_VALID),
    ('invalid/',    STAGING_INVALID),
    ('duplicates/', STAGING_DUPLICATES),
    ('outliers/',   STAGING_OUTLIERS),
]:
    files = list(folder_path.glob('*.parquet'))
    print(f'  staging/{folder_name}')
    if files:
        for f in files:
            size_kb = round(f.stat().st_size / 1024, 1)
            print(f'    {f.name:<45} ({size_kb:>8.1f} KB)')
    else:
        print(f'    (empty — no records of this type)')
    print()

**Explanation:**  
- **Valid records:** Pass all checks — proceed to transformation.  
- **Invalid records:** Have NULL in a key column. These are quarantined — never loaded to OLTP/OLAP.  
- **Duplicate records:** Secondary occurrences of the same primary key.  
- **Outlier records:** Statistically unusual values (IQR method). Kept in valid set but flagged for review.  
- All outputs are saved as Parquet for fast downstream reading.


---
## Section 3 — Data Transformation

**Objective:**  
Apply business rules and data cleaning to each table:

| Table | Key Transformations |
|-------|--------------------|
| Product | Type casting, Color standardization, Date parsing, Price flag |
| Customer | AccountNumber normalization, NULL fill for optional columns |
| Sales | Join Header + Detail on SalesOrderID, select required columns |


In [ ]:
print('Running transformation pipeline ...\n')
transformed = transform_all(val_result['valid'])

In [ ]:
# Inspect transformed Product
df_prod = transformed['product']
print('── Transformed Product — Schema ──')
data_cols = [c for c in df_prod.columns if not c.startswith('_')]
print(df_prod[data_cols].dtypes.to_string())

print('\n── Sample (first 5 rows) ──')
display(df_prod[['ProductID','Name','Color','StandardCost','ListPrice','SellStartDate']].head())

print(f'\nUnique Colors in cleaned Product:')
print(df_prod['Color'].value_counts().to_string())

In [ ]:
# Inspect transformed Sales (joined)
df_sales = transformed['sales']
print('── Transformed Sales (Header + Detail joined) ──')
print(f'Rows  : {len(df_sales):,}')
print(f'Columns: {len(df_sales.columns)}')
print()
display(df_sales[[
    'SalesOrderID','SalesOrderDetailID','ProductID','CustomerID',
    'OrderDate','OrderQty','UnitPrice','LineTotal'
]].head())

**Explanation:**  
- `StandardCost` and `ListPrice` are now `float64` — ready for arithmetic.  
- `SellStartDate` is now a proper `datetime64` — ready for time-series operations.  
- `Color` is title-cased and blank values replaced with `'N/A'` — standardized for grouping.  
- The Sales table is a **denormalized join** of Header + Detail — one row per line item,  
  combining order-level data (CustomerID, OrderDate) with line-level data (ProductID, OrderQty).


---
## Section 4 — Aggregations

**Objective:**  
Build summary datasets for analytics: top products, customer spending, sales trends.


In [ ]:
aggs = transformed['aggregations']

print('── Top 10 Products by Sales ──')
top10 = aggs['top10_products'][['Name','Color','TotalSales','TotalQty','OrderCount']]
top10['TotalSales'] = top10['TotalSales'].map('${:,.0f}'.format)
top10['TotalQty']   = top10['TotalQty'].map('{:,}'.format)
display(top10)

In [ ]:
print('── Sales by Year ──')
yr = aggs['sales_by_year'].copy()
yr['TotalSales'] = yr['TotalSales'].map('${:,.0f}'.format)
display(yr)

In [ ]:
print('── Sales by Month (most recent 12 months) ──')
mo = aggs['sales_by_month'].tail(12).copy()
mo['TotalSales'] = mo['TotalSales'].map('${:,.0f}'.format)
display(mo[['YearMonth','TotalSales']])

In [ ]:
print('── Top 10 Customers by Sales ──')
top_cust = aggs['sales_by_customer'].head(10).copy()
top_cust['TotalSales'] = top_cust['TotalSales'].map('${:,.0f}'.format)
display(top_cust[['CustomerID','TotalSales','TotalQty','OrderCount']])

**Explanation:**  
- `groupby().agg()` is the Pandas way to compute GROUP BY aggregations — equivalent to SQL `GROUP BY`.  
- We compute TotalSales (SUM), TotalQty (SUM), OrderCount (COUNT DISTINCT) per dimension.  
- Aggregations are saved as Parquet in `staging/valid/` for reuse by the dashboard notebook.  
- Top 10 by sales is the most commonly requested KPI in any retail analytics project.


---
## Section 5 — OLTP Schema Design

**Objective:**  
Understand the OLTP (Online Transaction Processing) schema and compare it to OLAP.

### What is OLTP?

| Characteristic | OLTP |
|---------------|------|
| Purpose | Record business transactions as they happen |
| Design | Normalized (3NF) — minimize data redundancy |
| Operations | Frequent INSERT, UPDATE, DELETE |
| Query type | Single-record lookups, small result sets |
| Joins | Many (normalized tables reference each other) |
| Example | A customer places an order → rows written to 2 tables |

### OLTP Relationships

```
product ←──────────────── salesorderdetail
                                │
customer ←── salesorderheader ──┘
```

**Foreign Key Rules:**
- Every `SalesOrderDetail.SalesOrderID` must exist in `SalesOrderHeader`
- Every `SalesOrderDetail.ProductID` must exist in `Product`
- Every `SalesOrderHeader.CustomerID` must exist in `Customer`


In [ ]:
# Demonstrate the OLTP schema using our transformed DataFrames
print('OLTP Schema — Record Counts (what would be loaded to PostgreSQL)\n')
print('  Table                  Records    Primary Key')
print('  ' + '-' * 55)
print(f'  product                {len(transformed["product"]):>7,}    ProductID')
print(f'  customer               {len(transformed["customer"]):>7,}    CustomerID')

# SalesOrderHeader: load from valid staging (before join)
hdr_path = STAGING_VALID / 'valid_salesorderheader.parquet'
if hdr_path.exists():
    df_hdr = pd.read_parquet(hdr_path)
    print(f'  salesorderheader       {len(df_hdr):>7,}    SalesOrderID')

dtl_path = STAGING_VALID / 'valid_salesorderdetail.parquet'
if dtl_path.exists():
    df_dtl = pd.read_parquet(dtl_path)
    print(f'  salesorderdetail       {len(df_dtl):>7,}    SalesOrderID + SalesOrderDetailID')

print()
print('  SQL DDL: see sql/oltp_schema.sql')

In [ ]:
# Demonstrate referential integrity check (FK validation)
print('── Referential Integrity Check ──')
print('(Simulating what the DB FOREIGN KEY constraint enforces)\n')

df_sales = transformed['sales']
valid_product_ids  = set(transformed['product']['ProductID'].dropna().astype(int))
valid_customer_ids = set(transformed['customer']['CustomerID'].dropna().astype(int))

orphan_product  = df_sales[~df_sales['ProductID'].isin(valid_product_ids)]
orphan_customer = df_sales[~df_sales['CustomerID'].isin(valid_customer_ids)]

print(f'  Sales rows with invalid ProductID  : {len(orphan_product):,}')
print(f'  Sales rows with invalid CustomerID : {len(orphan_customer):,}')

if len(orphan_product) == 0 and len(orphan_customer) == 0:
    print('\n  All foreign key references are valid.')
else:
    print('\n  FK violations detected — these rows would be rejected by the DB.')

**Explanation:**  
- In OLTP, **foreign keys** enforce data integrity at the database level.  
  A detail row cannot reference a product or order that doesn't exist.  
- We simulate this check in Pandas by comparing sets of IDs.  
- In `src/oltp.py`, the actual PostgreSQL tables enforce these constraints via `FOREIGN KEY` DDL.  
- The SQL DDL is in [`sql/oltp_schema.sql`](../sql/oltp_schema.sql).


---
## Section 6 — Star Schema (OLAP)

**Objective:**  
Build the Star Schema and understand the difference from OLTP.

### What is OLAP?

| Characteristic | OLAP |
|---------------|------|
| Purpose | Analyze large volumes of historical data |
| Design | Denormalized — fewer joins, faster aggregations |
| Operations | SELECT, GROUP BY, aggregations (no writes during analysis) |
| Query type | Large result sets, many rows |
| Joins | Few (fact → dimensions) |
| Example | "What were total sales per month per product category last year?" |

### Star Schema Structure

```
             DimDate
               │
DimProduct ──── FactSales ──── DimCustomer
```

- **Fact table** = what happened (one row per line item, numeric measures)
- **Dimension tables** = the context (who, what, when, where)
- **Surrogate keys** = integer keys generated by the DW (not from source system)


In [ ]:
print('Building Star Schema ...\n')
olap_data = build_olap(transformed)

In [ ]:
print('── DimDate — First 5 rows ──')
display(olap_data['dim_date'].head())
print(f'\nDimDate total rows: {len(olap_data["dim_date"]):,}')
print('Notice: Pre-computed Year, Quarter, Month, Week columns.')
print('These enable roll-up and drill-down without string parsing at query time.')

In [ ]:
print('── DimProduct — First 5 rows ──')
display(olap_data['dim_product'][[
    'product_key','product_id','name','color','product_line',
    'standard_cost','list_price'
]].head())
print(f'\nDimProduct: {len(olap_data["dim_product"]):,} products')
print('product_key = surrogate key (generated by DW)')
print('product_id  = natural key  (from source CSV)')

In [ ]:
print('── FactSales — First 5 rows ──')
display(olap_data['fact_sales'][[
    'fact_id','product_key','customer_key','date_key',
    'sales_order_id','order_qty','unit_price','line_total'
]].head())

fact = olap_data['fact_sales']
print(f'\nFactSales: {len(fact):,} rows')
print(f'Total Sales Value: ${fact["line_total"].sum():,.2f}')
print(f'Total Units Sold : {fact["order_qty"].sum():,}')
print(f'Unique Orders    : {fact["sales_order_id"].nunique():,}')

In [ ]:
# OLTP vs OLAP comparison
print('\n── OLTP vs OLAP Comparison ──\n')
comparison = {
    'Model':           ['OLTP (Normalized 3NF)', 'OLAP (Star Schema)'],
    'Purpose':         ['Record transactions', 'Analyze history'],
    'Tables':          ['4 (product, customer, header, detail)', '4 (DimDate, DimProduct, DimCustomer, FactSales)'],
    'Joins needed':    ['3 (header-detail, detail-product, header-customer)', '1–2 (fact-dimension)'],
    'Key type':        ['Natural key (ProductID from source)', 'Surrogate key (product_key from DW)'],
    'Optimized for':   ['INSERT/UPDATE/DELETE', 'SELECT/GROUP BY/aggregations'],
}
display(pd.DataFrame(comparison).set_index('Model').T)

**Explanation:**  
- **DimDate** is a pre-built calendar table. Instead of parsing dates at query time,  
  all date attributes (month, quarter, year) are pre-computed.  
- **Surrogate keys** (`product_key`, `customer_key`) are integers generated by the data warehouse.  
  They are stable even if the source `ProductID` changes.  
- **FactSales** contains only numbers (measures) and foreign keys (surrogate keys).  
  No descriptive text — that lives in the dimension tables.  
- The SQL DDL is in [`sql/star_schema.sql`](../sql/star_schema.sql).


---
## Section 7 — OLAP Operations

**Objective:**  
Demonstrate the four classic OLAP operations using Pandas.

| Operation | Description | Example |
|-----------|-------------|--------|
| **Roll-up** | Aggregate to coarser granularity | Daily → Monthly → Yearly |
| **Drill-down** | Expand to finer granularity | Year → Quarter → Month → Day |
| **Slice** | Filter one dimension to a single value | Sales where color = 'Black' |
| **Dice** | Filter multiple dimensions simultaneously | Black/Silver products, 2022–2024, online orders |


In [ ]:
fact = olap_data['fact_sales']
dim_date = olap_data['dim_date']
dim_product = olap_data['dim_product']
dim_customer = olap_data['dim_customer']

rollup = olap_rollup(fact, dim_date)

print('OLAP OPERATION 1: ROLL-UP')
print('Pattern: Daily → Monthly → Yearly\n')

print('  Yearly Sales (most coarse — highest level):')
yr = rollup['yearly'].copy()
yr['TotalSales'] = yr['TotalSales'].map('${:,.0f}'.format)
display(yr)

print('\n  Monthly Sales (sample — most recent 8 months):')
mo = rollup['monthly'].tail(8).copy()
mo['TotalSales'] = mo['TotalSales'].map('${:,.0f}'.format)
display(mo)

In [ ]:
drilldown = olap_drilldown(fact, dim_date)

print('OLAP OPERATION 2: DRILL-DOWN')
print('Pattern: Year → Quarter → Month → Day\n')

print('  Year level:')
display(drilldown['year'])

print('\n  Quarter level (first 8 rows):')
display(drilldown['quarter'].head(8))

print('\n  Explanation:')
print('  Drill-down lets analysts start with the big picture (year)')
print('  and "zoom in" to find the root cause of a trend.')

In [ ]:
sliced = olap_slice(fact, dim_product, dim_date, color_filter='Black')

print('OLAP OPERATION 3: SLICE')
print('Filter: Color = "Black" only\n')
print('  Monthly sales for Black products:')
sliced_display = sliced.copy()
sliced_display['TotalSales'] = sliced_display['TotalSales'].map('${:,.0f}'.format)
display(sliced_display)

print('\n  Explanation:')
print('  Slice fixes ONE dimension to a single value.')
print('  Like cutting a slice of bread — you see one cross-section of the data cube.')

In [ ]:
diced = olap_dice(
    fact, dim_product, dim_customer, dim_date,
    colors=['Black', 'Silver'],
    year_range=(2022, 2024),
    online_only=False,
)

print('OLAP OPERATION 4: DICE')
print('Filters: color IN (Black, Silver) AND year BETWEEN 2022-2024\n')
print('  Top 10 product-month combinations:')
diced_display = diced.head(10).copy()
diced_display['TotalSales'] = diced_display['TotalSales'].map('${:,.0f}'.format)
display(diced_display)

print('\n  Explanation:')
print('  Dice applies filters on MULTIPLE dimensions simultaneously.')
print('  Like cutting a small cube out of the larger data cube.')

In [ ]:
# Show the equivalent SQL for each operation
print('== Equivalent SQL Operations ==')
print('(See sql/analytics.sql for the full SQL versions)\n')

print('ROLL-UP (Daily -> Monthly):')
print('''
  SELECT d.year_month, SUM(f.line_total) AS monthly_sales
  FROM olap.fact_sales f
  JOIN olap.dim_date   d ON f.date_key = d.date_key
  GROUP BY d.year_month
  ORDER BY d.year_month;
''')

print('SLICE (color = Black):')
print('''
  SELECT d.year_month, SUM(f.line_total) AS total_sales
  FROM olap.fact_sales  f
  JOIN olap.dim_product  p ON f.product_key = p.product_key
  JOIN olap.dim_date     d ON f.date_key    = d.date_key
  WHERE p.color = 'Black'
  GROUP BY d.year_month;
''')

**Explanation:**  
- **Roll-up** reduces detail: `daily → monthly → yearly`. Each level is an aggregation of the one below it.  
- **Drill-down** is the reverse: start coarse, go finer. Analysts use this to investigate anomalies.  
- **Slice** is a single-value filter on one dimension (like SQL `WHERE color = 'Black'`).  
- **Dice** combines multiple filters (like SQL `WHERE color IN ('Black','Silver') AND year BETWEEN ...`).  
- All four operations are enabled by the Star Schema structure: the DimDate pre-computed hierarchy  
  makes roll-up/drill-down trivial, and dimension attributes make slicing/dicing fast.


---
## Section 8 — Step 2 Summary


In [ ]:
fact = olap_data['fact_sales']
val_rep = val_result['reports']

total_outliers = sum(r['outliers']['outlier_records'] for r in val_rep.values())
total_invalid  = sum(r['nulls']['null_records'] for r in val_rep.values())
total_dupes    = sum(r['duplicates']['duplicate_records'] for r in val_rep.values())

print('=' * 60)
print('  STEP 2 COMPLETE — SCHEMA DESIGN SUMMARY')
print('=' * 60)
print()
print('  VALIDATION RESULTS:')
print(f'    Total source records : {sum(len(df) for df in raw.values()):>10,}')
print(f'    Invalid (null key)   : {total_invalid:>10,}')
print(f'    Duplicate records    : {total_dupes:>10,}')
print(f'    Outlier records      : {total_outliers:>10,}')
print()
print('  TRANSFORMATION RESULTS:')
print(f'    Product records      : {len(transformed["product"]):>10,}')
print(f'    Customer records     : {len(transformed["customer"]):>10,}')
print(f'    Sales records        : {len(transformed["sales"]):>10,}')
print()
print('  STAR SCHEMA:')
print(f'    DimDate rows         : {len(olap_data["dim_date"]):>10,}')
print(f'    DimProduct rows      : {len(olap_data["dim_product"]):>10,}')
print(f'    DimCustomer rows     : {len(olap_data["dim_customer"]):>10,}')
print(f'    FactSales rows       : {len(fact):>10,}')
print(f'    Total Sales Value    : ${fact["line_total"].sum():>19,.2f}')
print()
print('  SQL FILES GENERATED:')
print('    sql/oltp_schema.sql  — OLTP DDL (product, customer, orders)')
print('    sql/star_schema.sql  — Star Schema DDL (DimDate, DimProduct, DimCustomer, FactSales)')
print('    sql/analytics.sql    — OLAP operations (Roll-up, Drill-down, Slice, Dice)')
print()
print('  Next: Step 3 — Python Batch Pipeline (run_pipeline.py)')
print('=' * 60)

---
## Concepts Learned in This Notebook

| Concept | Definition |
|---------|------------|
| **Schema Validation** | Checking that expected columns are present before processing |
| **Null Validation** | Key columns must not be empty — records with null keys are quarantined |
| **IQR Outlier Detection** | `Q1 - 1.5×IQR` to `Q3 + 1.5×IQR` defines the normal range; values outside are outliers |
| **Data Transformation** | Type casting, string normalization, NULL filling, joining tables |
| **OLTP** | Normalized operational database — captures business transactions |
| **OLAP** | Denormalized analytical database — optimized for fast aggregations |
| **Star Schema** | One Fact table surrounded by Dimension tables |
| **Fact Table** | Stores numeric measures (line_total, order_qty) + surrogate foreign keys |
| **Dimension Table** | Stores descriptive attributes (product name, customer account, date hierarchy) |
| **Surrogate Key** | Integer key generated by the data warehouse (not from source) |
| **Roll-up** | Aggregating from fine grain to coarse grain (day → month → year) |
| **Drill-down** | Moving from coarse grain to fine grain (year → quarter → month → day) |
| **Slice** | Filter one dimension to a single value |
| **Dice** | Filter multiple dimensions simultaneously |

---
*Next notebook:* `03_batch_pipeline.ipynb` — Python Batch Pipeline with run_pipeline()  
*Developed with:* **Antigravity** — AI Coding Assistant
